# CEP lab runs on Kaggle (development environment)

Runs the same harness as the laptop and AWS (design.md §7) with Ollama on Kaggle's free GPU.
Nothing in the experiment changes: same models (Ollama files), same trials, temperature 0, same JSONL logs.

**Before running:** Settings → Accelerator = **GPU T4 x2** (or P100), Internet = **On** (needs a phone-verified account).

**How to use**
1. Edit the CONFIG cell (MODE, MODELS, DEFENSE, ...).
2. Short jobs: Run All. Long jobs: *Save Version → Save & Run All (Commit)*, which keeps running after you close the tab (max 12 h per session).
3. Download `results_<ENV>.zip` from the Output tab, unzip it in the repo root, commit `results/` (and `reports/pilot_*.md`).
4. To continue an interrupted run in a new session: upload the previous zip as a Kaggle Dataset, attach it (Add Input), re-run. Finished trials are skipped.

In [ ]:
# ---------------- CONFIG ----------------
REPO_URL = "https://github.com/MuhammadTalha57/CEP.git"
BRANCH = "master"
MODE = "run"                # "pilot" (timing test) | "run" (experiments)
MODELS = ["qwen3-1.7b", "llama3.2-3b", "gemma3-1b", "qwen3-4b-instruct"]   # ids from configs/models.yaml
ENV = "kaggle"              # results/<ENV>/...; pilot uses ENV + "-pilot"
DEFENSE = "none"            # MODE="run": "none" (baseline), "all" (every defense in experiment.yaml), or one name
APPS = []                   # MODE="run": [] = all apps, or e.g. ["chatbot", "rag"]
EXTRA_ARGS = ""             # MODE="run": e.g. "--variants 10" or "--cells T2@C0 U"
WORKERS = 1                 # keep 1: identical conditions to AWS (OLLAMA_NUM_PARALLEL=1)

In [ ]:
import os, subprocess, time, json, glob, shutil, urllib.request
REPO = "/tmp/CEP"                       # code lives in /tmp; only the results zip goes to /kaggle/working
OUT = "/kaggle/working"
os.chdir(OUT)   # a re-run must not start inside a deleted folder ("getcwd() failed")
def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=OUT)
    print(r.stdout[-4000:], flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed ({r.returncode}): {cmd}")
    return r.stdout
sh("nvidia-smi --query-gpu=name,memory.total --format=csv", check=False)

## 1. Install and start Ollama

In [ ]:
sh("apt-get -qq update && apt-get -qq install -y zstd pciutils lshw > /dev/null", check=False)
sh("curl -fsSL https://ollama.com/install.sh | sh")
os.environ["OLLAMA_KEEP_ALIVE"] = "24h"     # same settings as docker-compose.yml
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
subprocess.Popen("nohup ollama serve > /tmp/ollama.log 2>&1 &", shell=True)
for _ in range(60):
    try:
        print(urllib.request.urlopen("http://localhost:11434/api/version", timeout=2).read().decode()); break
    except Exception:
        time.sleep(2)
else:
    print(open("/tmp/ollama.log").read()[-3000:]); raise RuntimeError("ollama did not start")

## 2. Get the code

In [ ]:
os.chdir(OUT)
if os.path.isdir(f"{REPO}/.git"):
    # re-run in the same session: update the code but KEEP results/ from earlier runs
    sh(f"git -C {REPO} fetch --depth 1 origin {BRANCH} && git -C {REPO} reset --hard FETCH_HEAD")
else:
    shutil.rmtree(REPO, ignore_errors=True)
    sh(f"git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}")
sh(f"pip install -q -r {REPO}/requirements.txt && pip install -q -e {REPO}")
sh(f"cd {REPO} && python -m pytest -q", check=False)   # mock-model tests; must pass
COMMIT = sh(f"git -C {REPO} rev-parse --short HEAD").strip()

## 3. Pull the models (same Ollama files as AWS)

In [ ]:
import yaml
specs = {m["id"]: m for m in yaml.safe_load(open(f"{REPO}/configs/models.yaml"))["models"]}
for mid in MODELS:
    sh(f"ollama pull {specs[mid]['model']}")
sh("ollama list")

## 4. Restore results from a previous session (optional)

In [ ]:
restored = 0
for z in glob.glob("/kaggle/input/**/results_*.zip", recursive=True):
    print("restoring", z)
    sh(f"cd {REPO} && unzip -o -q '{z}'")
    restored += 1
# Kaggle may auto-extract an uploaded zip into folders: then copy its results/ folder instead
for d in glob.glob("/kaggle/input/**/results", recursive=True):
    if os.path.isdir(d):
        print("restoring folder", d)
        sh(f"cp -r '{d}/.' {REPO}/results/")
        restored += 1
n_files = len(glob.glob(f"{REPO}/results/{ENV}/**/*.jsonl", recursive=True))
print(f"restored {restored} input(s); {n_files} result files now in results/{ENV}/")
if MODE == "run" and DEFENSE != "none" and n_files == 0:
    raise RuntimeError("no earlier results found: attach the results dataset (Add Input) before a defense run, "
                       "otherwise the baseline would be re-run")

## 5. Record the environment

In [ ]:
env_info = {
    "env": ENV, "commit": COMMIT, "mode": MODE, "models": MODELS,
    "gpu": sh("nvidia-smi --query-gpu=name --format=csv,noheader", check=False).strip(),
    "ollama": sh("ollama --version", check=False).strip(),
    "python": sh("python --version").strip(),
    "ollama_models": sh("ollama list", check=False).strip().splitlines(),   # tags + digests (tags can move)
    "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
os.makedirs(f"{REPO}/results/{ENV}", exist_ok=True)
with open(f"{REPO}/results/{ENV}/environment_{MODE}_{int(time.time())}.json", "w") as f:
    json.dump(env_info, f, indent=2)
print(env_info)

## 6. Run

In [ ]:
os.chdir(REPO)
models = " ".join(MODELS)
if MODE == "pilot":
    cmd = f"python -m llmsec.harness.pilot --models {models} --env {ENV}-pilot"
else:
    apps = f"--apps {' '.join(APPS)}" if APPS else ""
    cmd = f"python -m llmsec.harness.run --models {models} --env {ENV} --defense {DEFENSE} {apps} --workers {WORKERS} {EXTRA_ARGS}"
# stream output live (long runs)
p = subprocess.Popen(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
for line in p.stdout:
    print(line, end="", flush=True)
print("exit code", p.wait())

## 7. Summary and download

In [ ]:
os.chdir(REPO)
env_dir = f"{ENV}-pilot" if MODE == "pilot" else ENV
if os.path.isdir(f"results/{env_dir}"):
    sh(f"python -m llmsec.harness.report results/{env_dir}", check=False)
import zipfile
zip_path = f"{OUT}/results_{env_dir}.zip"
files = [f for d in {f"results/{env_dir}", f"results/{ENV}"} for f in glob.glob(f"{d}/**/*", recursive=True)
         if os.path.isfile(f)] + glob.glob(f"reports/pilot_{env_dir}.md")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(set(files)):
        z.write(f)            # paths relative to the repo root: unzip in the repo root
print(len(files), "files ->", zip_path, "(download from the Output tab)")